# Fund Analysis
This notebook analyzes fund performance relative to the SP500.

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.regression.rolling import RollingOLS
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

# Load data
df = pd.read_csv('../data/concat_hf_data.csv')
if 'date' in df.columns:
    df['date'] = pd.to_datetime(df['date'])
    df = df.set_index('date')

# Keep only sp500 and target fund columns
fund_col = 'gator'
df = df[['sp500', fund_col]].dropna()

sp500 = df['sp500']
fund = df[fund_col]

df.head()

In [ ]:
# Dataset Info and Hit Rates
start_date = df.index.min()
end_date = df.index.max()
n_months = len(df)
time_elapsed = end_date - start_date
years_elapsed = time_elapsed.days / 365.25

print("--- Dataset Info ---")
print(f"Start Date: {start_date.date()}")
print(f"End Date:   {end_date.date()}")
print(f"Time Elapsed: {time_elapsed.days} days ({years_elapsed:.2f} years)")
print(f"Total Data Points: {n_months} months")

fund_pos_months = (fund > 0).sum()
sp500_pos_months = (sp500 > 0).sum()
print("\n--- Positive Months (Hit Rates) ---")
print(f"Fund Positive Months:  {fund_pos_months/n_months:.2%} ({fund_pos_months}/{n_months})")
print(f"SP500 Positive Months: {sp500_pos_months/n_months:.2%} ({sp500_pos_months}/{n_months})")

sp500_neg_mask = sp500 < 0
sp500_pos_mask = sp500 > 0
n_sp500_neg = sp500_neg_mask.sum()
n_sp500_pos = sp500_pos_mask.sum()

fund_pos_when_sp500_neg = (fund[sp500_neg_mask] > 0).sum()
fund_neg_when_sp500_pos = (fund[sp500_pos_mask] < 0).sum()
print("\n--- Conditional Hit Rates ---")
print(f"Fund Positive when SP500 Negative: {fund_pos_when_sp500_neg / n_sp500_neg if n_sp500_neg > 0 else np.nan:.2%} ({fund_pos_when_sp500_neg}/{n_sp500_neg})")
print(f"Fund Negative when SP500 Positive: {fund_neg_when_sp500_pos / n_sp500_pos if n_sp500_pos > 0 else np.nan:.2%} ({fund_neg_when_sp500_pos}/{n_sp500_pos})")

In [ ]:
# Trading Metrics
mean_ret = fund.mean() * 12
vol = fund.std() * np.sqrt(12)
sharpe = mean_ret / vol if vol != 0 else np.nan

# CAGR
cum_ret = (1 + fund).prod()
n_years = n_months / 12
cagr = cum_ret**(1/n_years) - 1

# Max Drawdown
cum_wealth = (1 + fund).cumprod()
peak = cum_wealth.cummax()
drawdown = (cum_wealth - peak) / peak
max_drawdown = drawdown.min()

# Calmar Ratio
calmar = cagr / abs(max_drawdown) if max_drawdown != 0 else np.nan

print(f"--- Trading Metrics (n={n_months} months) ---")
print(f"Annualized Mean Return: {mean_ret:.4f}")
print(f"Annualized Volatility:  {vol:.4f}")
print(f"Sharpe Ratio:           {sharpe:.4f}")
print(f"CAGR:                   {cagr:.4f}")
print(f"Max Drawdown:           {max_drawdown:.4f}")
print(f"Calmar Ratio:           {calmar:.4f}")

In [ ]:
# Joint Metrics
pearson_corr = fund.corr(sp500, method='pearson')
spearman_corr = fund.corr(sp500, method='spearman')

# Bootstrap for correlations (1000 iterations, 95% CI -> 5% safety interval)
def bootstrap_corr(x, y, method='pearson', block_size=1, iters=1000):
    np.random.seed(42) # For reproducibility
    n = len(x)
    corrs = []
    indices = np.arange(n)
    for _ in range(iters):
        if block_size == 1:
            idx = np.random.choice(indices, size=n, replace=True)
        else:
            num_blocks = int(np.ceil(n / block_size))
            block_starts = np.random.choice(n - block_size + 1, size=num_blocks, replace=True)
            idx = [i for start in block_starts for i in range(start, start + block_size)][:n]
        x_samp = x.iloc[idx]
        y_samp = y.iloc[idx]
        corrs.append(x_samp.corr(y_samp, method=method))
    return np.nanpercentile(corrs, 2.5), np.nanpercentile(corrs, 97.5)

pearson_std_ci = bootstrap_corr(fund, sp500, method='pearson', block_size=1)
pearson_blk_ci = bootstrap_corr(fund, sp500, method='pearson', block_size=3)
spearman_std_ci = bootstrap_corr(fund, sp500, method='spearman', block_size=1)
spearman_blk_ci = bootstrap_corr(fund, sp500, method='spearman', block_size=3)

X = sm.add_constant(sp500)
model = sm.OLS(fund, X).fit()
alpha = model.params['const'] * 12
beta = model.params['sp500']
alpha_tstat = model.tvalues['const']
beta_tstat = model.tvalues['sp500']

print(f"--- Joint Metrics (n={n_months} months) ---")
print(f"Pearson Correlation:  {pearson_corr:.4f}")
print(f"  - 95% CI (Standard Bootstrap): [{pearson_std_ci[0]:.4f}, {pearson_std_ci[1]:.4f}]")
print(f"  - 95% CI (Block=3 Bootstrap):  [{pearson_blk_ci[0]:.4f}, {pearson_blk_ci[1]:.4f}]")
print(f"Spearman Correlation: {spearman_corr:.4f}")
print(f"  - 95% CI (Standard Bootstrap): [{spearman_std_ci[0]:.4f}, {spearman_std_ci[1]:.4f}]")
print(f"  - 95% CI (Block=3 Bootstrap):  [{spearman_blk_ci[0]:.4f}, {spearman_blk_ci[1]:.4f}]")
print(f"Regression Alpha (Annualized): {alpha:.4f} (t-stat: {alpha_tstat:.4f})")
print(f"Regression Beta:               {beta:.4f} (t-stat: {beta_tstat:.4f})")

In [ ]:
# Upside / Downside Capture
up_months = df[sp500_pos_mask]
down_months = df[sp500_neg_mask]

up_capture = up_months[fund_col].mean() / up_months['sp500'].mean()
down_capture = down_months[fund_col].mean() / down_months['sp500'].mean()

print("--- Capture Ratios ---")
print(f"Upside Capture:   {up_capture:.4f} (n={len(up_months)})")
print(f"Downside Capture: {down_capture:.4f} (n={len(down_months)})")

In [ ]:
# Conditional Metrics: Positive / Negative SP500 Months
model_up = sm.OLS(up_months[fund_col], sm.add_constant(up_months['sp500'])).fit()
print(f"--- Up Months Metrics (n={len(up_months)}) ---")
print(f"Up Alpha (Annualized): {model_up.params['const'] * 12:.4f} (t-stat: {model_up.tvalues['const']:.4f})")
print(f"Up Beta:               {model_up.params['sp500']:.4f} (t-stat: {model_up.tvalues['sp500']:.4f})")

print(f"\n--- Down Months Metrics (n={len(down_months)}) ---")
model_down = sm.OLS(down_months[fund_col], sm.add_constant(down_months['sp500'])).fit()
print(f"Down Alpha (Annualized): {model_down.params['const'] * 12:.4f} (t-stat: {model_down.tvalues['const']:.4f})")
print(f"Down Beta:               {model_down.params['sp500']:.4f} (t-stat: {model_down.tvalues['sp500']:.4f})")

In [ ]:
# Correlation when SP500 is lower than 1st decile and 1st quartile
decile_1 = sp500.quantile(0.10)
df_d1 = df[df['sp500'] < decile_1]
d1_pearson = df_d1[fund_col].corr(df_d1['sp500'], method='pearson')
d1_spearman = df_d1[fund_col].corr(df_d1['sp500'], method='spearman')

quartile_1 = sp500.quantile(0.25)
df_q1 = df[df['sp500'] < quartile_1]
q1_pearson = df_q1[fund_col].corr(df_q1['sp500'], method='pearson')
q1_spearman = df_q1[fund_col].corr(df_q1['sp500'], method='spearman')

print("--- Tail Correlations ---")
print(f"SP500 < 1st Decile   | Pearson: {d1_pearson:.4f}, Spearman: {d1_spearman:.4f} (n={len(df_d1)})")
print(f"SP500 < 1st Quartile | Pearson: {q1_pearson:.4f}, Spearman: {q1_spearman:.4f} (n={len(df_q1)})")

In [ ]:
# Scatter plot
plt.figure(figsize=(10, 6))
plt.scatter(sp500, fund, alpha=0.7, color='steelblue')
plt.axvline(0, color='grey', linestyle='--', linewidth=1)
plt.axhline(0, color='grey', linestyle='--', linewidth=1)
plt.xlabel('SP500 Returns')
plt.ylabel(f'{fund_col} Returns')
plt.title(f'Scatter plot of {fund_col} Returns vs SP500 Returns')

# Regression line
x_vals = np.linspace(sp500.min(), sp500.max(), 100)
y_vals = model.params['sp500'] * x_vals + model.params['const']
plt.plot(x_vals, y_vals, color='firebrick', linewidth=2, label=f"Fit: y = {model.params['sp500']:.2f}x + {model.params['const'] * 12:.4f} (Ann. Alpha)")

plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Rolling Metrics (12-month and 24-month)
roll_corr_12 = fund.rolling(12).corr(sp500)
roll_corr_24 = fund.rolling(24).corr(sp500)

X_roll = sm.add_constant(sp500)
roll_model_12 = RollingOLS(fund, X_roll, window=12).fit()
roll_alpha_12 = roll_model_12.params['const'] * 12
roll_beta_12 = roll_model_12.params['sp500']

roll_model_24 = RollingOLS(fund, X_roll, window=24).fit()
roll_alpha_24 = roll_model_24.params['const'] * 12
roll_beta_24 = roll_model_24.params['sp500']

fig, axes = plt.subplots(3, 1, figsize=(12, 18), sharex=True)

# Rolling Correlation
axes[0].plot(roll_corr_12, label='12-Month', color='steelblue')
axes[0].plot(roll_corr_24, label='24-Month', color='firebrick')
axes[0].axhline(0, color='grey', linestyle='--')
axes[0].set_title(f'Rolling Correlation between {fund_col} and SP500')
axes[0].legend()

# Rolling Beta
axes[1].plot(roll_beta_12, label='12-Month', color='steelblue')
axes[1].plot(roll_beta_24, label='24-Month', color='firebrick')
axes[1].axhline(1, color='grey', linestyle='--')
axes[1].axhline(0, color='grey', linestyle=':')
axes[1].set_title(f'Rolling Beta of {fund_col} relative to SP500')
axes[1].legend()

# Rolling Alpha
axes[2].plot(roll_alpha_12, label='12-Month', color='steelblue')
axes[2].plot(roll_alpha_24, label='24-Month', color='firebrick')
axes[2].axhline(0, color='grey', linestyle='--')
axes[2].set_title(f'Rolling Annualized Alpha of {fund_col} relative to SP500')
axes[2].legend()

plt.tight_layout()
plt.show()